# 环节 04 · 信赖域与 PPO（配套 Notebook）

> 配套长文：[环节04-信赖域与PPO详解.md](./环节04-信赖域与PPO详解.md)
> 定位：把 **clip 的悲观下界**画出来、定位**梯度截断点**、量化**重要性比的方差爆炸**、演示 **KL 早停**。全部**纯 Python 标准库**。

**怎么跑**

- 依赖：无。§2 的输出就是长文 §8 的两张表。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 重要性采样 | §2 | 序列级 ρ 随长度指数爆炸 |
| §2 clip 两张表 | §4 / §8 | Â>0 拦上界、Â<0 拦下界（且只取消激励） |
| §3 梯度截断点 | §4 | `min` 选中哪一支，梯度就是哪一支 |
| §4 clipfrac | §7 | 被截断样本占比的监控阈值 |
| §5 KL 早停 | §7 | KL 超目标就停止更新 |


## 1. 重要性采样：让"旧数据"服务"新策略"

`ρ = π_new(a|s) / π_old(a|s)`。对**整条序列**，ρ 是所有步的连乘 —— 于是**方差随长度指数增长**。这就是 PPO 宁愿把每批数据只更新几次、把自己限制成近似 on-policy 的根本原因。


In [ ]:
print("序列级重要性比 ρ = ∏_t ρ_t 的增长：")
for T, per in ((10, 1.1), (50, 1.1), (100, 1.05), (200, 1.02)):
    print(f"  长度 T = {T:<4} 每步 ρ = {per}   →   序列 ρ = {per ** T:.3e}")
print("→ 只要每步略微偏离，长度一长就连乘到天文数字：所以必须 clip。")


## 2. clip 到底在哪截断（ε = 0.2）

`L^CLIP = min( ρÂ , clip(ρ, 0.8, 1.2)·Â )`

- `Â > 0`（好动作）：ρ 超过 1.2 后，目标被钉死、**梯度变 0** —— "猛推好动作"失去额外收益；
- `Â < 0`（坏动作）：ρ 低于 0.8 时被钉在 `clip·Â`（**更负**的那支），同样是**梯度 0**；而 ρ 变大（本来就让目标更负）**天然是惩罚，不用 clip**。


In [ ]:
EPS = 0.2
clip = lambda x, lo, hi: max(lo, min(hi, x))

def L_clip(rho, A):
    unclipped = rho * A
    clipped = clip(rho, 1 - EPS, 1 + EPS) * A
    val = min(unclipped, clipped)
    grad = A if val == unclipped else 0.0          # min 选中哪一支，梯度就是哪一支
    return unclipped, val, grad

for A in (1.0, -1.0):
    print(f"===== Â = {A:+.1f} =====")
    for rho in (0.5, 0.8, 1.0, 1.2, 1.5, 2.0):
        u, v, gr = L_clip(rho, A)
        flag = " ← 梯度被截断" if gr == 0 else ""
        print(f"  ρ={rho:.1f}  L^CPI={u:+.2f}  L^CLIP={v:+.2f}  dL/dρ={gr:+.1f}{flag}")


## 3. 三处必须看懂的结论

1. `Â>0, ρ=1.5/2.0`：`L^CLIP` 钉死在 **1.20**、梯度 **0** —— 信任域用一个 `min` 就实现了。
2. `Â<0, ρ=0.5/0.7`：钉在 **−0.80**（比 `−0.50` **更负**），梯度也是 **0**。
3. `Â<0, ρ=1.5/2.0` **没被截断**（梯度仍是 −1）—— 因为 ρ 变大本身就让目标更负，**天然是惩罚**。很多面试者在这里答错。


## 4. clipfrac：上线必盯的监控量

`clipfrac` = 这一批里 ρ 被截断的样本比例。**长期 > 20%** 说明"步子太大"（新旧策略差太远）→ 该降 ε 或减少每批更新轮数。


In [ ]:
import math
import random

random.seed(3)
rhos = [math.exp(random.gauss(0, 0.25)) for _ in range(1000)]     # 一批重要性比
frac = sum(1 for r in rhos if r < 1 - EPS or r > 1 + EPS) / len(rhos)
print(f"这批数据的 clipfrac = {frac:.1%}")
print(f"阈值：> 20% 就该降 ε 或 n_epochs；当前 = {frac:.1%} → "
      + ("偏大，注意策略突变" if frac > 0.2 else "正常"))


## 5. KL 早停：别再改这批数据了

当 `KL(π_new ‖ π_old)` 超过目标值时，说明这批 rollout 已经被"榨干"，继续更新只会把策略推离数据分布 → **停止更新、重采数据**。


In [ ]:
def kl_early_stop(kl, target=0.02):
    if kl > 1.5 * target:
        return "立即停止（KL 已超目标的 1.5 倍）"
    if kl > target:
        return "本批更新完就停，重采数据"
    return "继续更新"

for kl in (0.005, 0.015, 0.025, 0.040):
    print(f"  KL = {kl:.3f}  →  {kl_early_stop(kl)}")


## 6. 小结与下钻

- **clip 是一个"悲观下界"**：让"把策略改远"完全失去额外收益（不管是好动作还是坏动作）。
- **两个不对称**：`Â>0` 拦上界 `1+ε`；`Â<0` 拦下界 `1−ε`，但 ρ 变大不拦（本来就惩罚）。
- **PPO 是近似 on-policy**：一批数据只更新几次，靠 ρ 修正 + clip + KL 早停三件套控住步子。

下一站：[环节 05 · 偏好对齐与 RLHF](./环节05-偏好对齐与RLHF详解.md)（奖励从"人类偏好"来）。
